# Feature B — Automatic checkpointed sentiment analysis

No manual labelling is used. Rating-derived labels are treated as **silver labels**, so reported scores are agreement measures—not human-validated accuracy. A multilingual pilot must pass before the full run. Pilot labels are reused by the full checkpoint.

In [ ]:
!pip install -q google-genai python-dotenv pyarrow tqdm scikit-learn matplotlib seaborn

In [ ]:
import os,sys
from pathlib import Path
from getpass import getpass
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix,cohen_kappa_score
from google.colab import drive
drive.mount('/content/drive')
PROJECT=Path('/content/drive/MyDrive/Amazon Mini Projects/Amazon-Marketplace-Product-Intelligence')
assert PROJECT.exists(),f'Wrong project path: {PROJECT}'; sys.path.insert(0,str(PROJECT))
from analysis.cleaning import load_clean
from analysis.labelling_model2 import GeminiLabeller,resolve_model_name
from analysis.sentiment_checkpointed import prepare_reviews,label_with_checkpoint,sentiment_summaries
if not os.getenv('GEMINI_API_KEY'): os.environ['GEMINI_API_KEY']=getpass('Gemini API key: ')
CHECKPOINT_DIR=PROJECT/'data'/'checkpoints'/'ReviewSentiment'
OUTPUT_DIR=PROJECT/'data'/'models'/'ReviewSentiment'; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
MODEL=resolve_model_name('gemini-3.5-flash-lite',('gemini-3.1-flash-lite','gemini-2.5-flash-lite'))
PROMPT_VERSION='sentiment_v2'; print('Selected model:',MODEL)

## 1. Preprocessing, EDA, and interpretation

The cleaned schema uses `rating`. All languages remain in scope. Ratings create automatic silver labels: 1–2 negative, 3 neutral, 4–5 positive. This is an imperfect proxy because text and rating can disagree.

In [ ]:
products,reviews=load_clean(); df=prepare_reviews(reviews)
print('Products:',len(products),'Reviews:',len(df),'Duplicate IDs:',df.review_id.duplicated().sum())
print('English share:',f"{df.language.eq('en').mean():.1%}"); display(df); display(df.isna().mean().mul(100).sort_values(ascending=False).to_frame('missing_percent'))
fig,axes=plt.subplots(1,3,figsize=(18,5))
sns.countplot(data=df,y='language',order=df.language.value_counts().head(15).index,ax=axes[0]); axes[0].set_title('Top languages')
sns.countplot(data=df,x='rating',ax=axes[1]); axes[1].set_title('Rating distribution')
sns.histplot(df.body_clean.str.len(),bins=50,ax=axes[2]); axes[2].set_title('Review length')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'sentiment_eda.png',dpi=160,bbox_inches='tight'); plt.show()
print('EDA decision: native multilingual Gemini; language-specific checks; log every failure; no translation unless later automatic evidence shows a problem.')

## 2. Multilingual stratified pilot

The pilot samples English/non-English reviews across all available silver sentiment classes. It uses the final checkpoint directory, so successful pilot labels are automatically reused in the complete run.

In [ ]:
parts=[]
for _,group in df[df.star_sentiment.notna()].groupby(['language_group','star_sentiment']):
    parts.append(group.sample(min(20,len(group)),random_state=42))
pilot=pd.concat(parts).drop_duplicates('review_id').reset_index(drop=True)
print('Pilot size:',len(pilot)); display(pd.crosstab(pilot.language_group,pilot.star_sentiment))
labeller=GeminiLabeller(model_name=MODEL,batch_size=10,request_delay=6.5)
assert labeller.test_connection(), 'Connection failed; stop and read the API error.'
pilot_labelled,checkpoint=label_with_checkpoint(pilot,labeller,CHECKPOINT_DIR,PROMPT_VERSION,batch_size=10)
print('Checkpoint:',checkpoint)

## 3. Automatic validation and language checks

Macro-F1, agreement, Cohen’s kappa, API success, and English/non-English results are reported against rating-based silver labels. They must not be called human accuracy.

In [ ]:
valid=pilot_labelled[pilot_labelled.sentiment.isin(['positive','neutral','negative']) & pilot_labelled.star_sentiment.notna()].copy()
success_rate=pilot_labelled.sentiment.isin(['positive','neutral','negative']).mean()
metrics={'api_success_rate':success_rate,'silver_agreement':accuracy_score(valid.star_sentiment,valid.sentiment),'silver_macro_f1':f1_score(valid.star_sentiment,valid.sentiment,average='macro',zero_division=0),'cohen_kappa':cohen_kappa_score(valid.star_sentiment,valid.sentiment)}
display(pd.Series(metrics).to_frame('value')); print(classification_report(valid.star_sentiment,valid.sentiment,labels=['positive','neutral','negative'],zero_division=0))
rows=[]
for name,g in valid.groupby('language_group'):
 rows.append({'language_group':name,'reviews':len(g),'agreement':accuracy_score(g.star_sentiment,g.sentiment),'macro_f1':f1_score(g.star_sentiment,g.sentiment,average='macro',zero_division=0)})
language_check=pd.DataFrame(rows); display(language_check)
cm=confusion_matrix(valid.star_sentiment,valid.sentiment,labels=['positive','neutral','negative']); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['positive','neutral','negative'],yticklabels=['positive','neutral','negative']); plt.xlabel('Gemini'); plt.ylabel('Rating silver label'); plt.show()
PILOT_PASS=success_rate>=0.95 and len(valid)>=30
if not PILOT_PASS: raise RuntimeError('Pilot did not pass API completeness checks. Do not start the full run.')
print('Pilot passed. Silver-label disagreement is expected and must be documented.')

## 4. Full resumable labelling

Run this cell repeatedly if quota stops the job. It reuses pilot and prior full labels only when review ID, body hash, model, and prompt version match. A completely failed batch stops the run safely.

In [ ]:
full_labeller=GeminiLabeller(model_name=MODEL,batch_size=10,request_delay=6.5)
labelled_all,checkpoint=label_with_checkpoint(df,full_labeller,CHECKPOINT_DIR,PROMPT_VERSION,batch_size=10)
completed=labelled_all.sentiment.isin(['positive','neutral','negative']).sum(); print('Completed:',completed,'/',len(labelled_all)); print('Checkpoint:',checkpoint)
if completed<len(labelled_all): print('Safe partial run: rerun this cell later. Finished reviews will not be requested again.')

## 5. Product and category summaries

Creates counts, percentages, and dominant sentiment for every product/category represented by successfully labelled reviews.

In [ ]:
usable=labelled_all[labelled_all.sentiment.isin(['positive','neutral','negative'])].copy()
product_summary,category_summary=sentiment_summaries(usable)
usable.to_parquet(OUTPUT_DIR/'reviews_with_sentiment.parquet',index=False)
product_summary.to_csv(OUTPUT_DIR/'sentiment_by_product.csv',index=False); category_summary.to_csv(OUTPUT_DIR/'sentiment_by_category.csv',index=False)
display(product_summary); display(category_summary); print('Saved:',OUTPUT_DIR)

## Reporting limitation

Because manual labelling was explicitly excluded, this project reports automatic reliability checks and silver-label agreement—not human-validated accuracy. If the marking rubric strictly requires manually checked accuracy, that particular requirement remains unmet.